In [1]:
import random
from collections import deque
 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
 
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
 
import gymnasium as gym
import sys
import os

sys.path.append(os.path.abspath(".."))
import gym_anytrading

In [2]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim: int,n_actions: int, hidden: int = 128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(obs_dim,hidden),
            nn.Tanh(),
            nn.Linear(hidden,hidden),
            nn.Tanh(),
        )
        self.actor = nn.Linear(hidden,n_actions)
        self.critic = nn.Linear(hidden,1)
    
    def forward(self,x:torch.tensor):
        features = self.backbone(x)
        logits = self.actor(features)
        value = self.critic(features)
        return logits, value

    def get_action(self,state:np.ndarray):
        state_t= torch.FloatTensor(state)
        logits, value = self(state_t)
        dist = Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return int(action.item()), float(log_prob.item()), float(value.item())

    def evaluate(self, states:torch.Tensor, actions: torch.Tensor):
        logits, values = self(states)
        dist = Categorical(logits=logits)
        log_probs = dist.log_prob(actions)
        entropy = dist.entropy()
        return log_probs, values.squeeze(-1), entropy

In [8]:
class RolloutBuffer:
    def __init__(self):
        self.clear()
    
    def clear(self):
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.dones = []
        self.values = []
    
    def push(self, state,action,reward,log_prob,value, done):
        self.states.append(np.array(state))
        self.actions.append(float(action))
        self.rewards.append(float(reward))
        self.log_probs.append(float(log_prob))
        self.values.append(float(value))
        self.dones.append(bool(done))
    
    def get(self):
        return (
            torch.FloatTensor(np.array(self.states)),
            torch.LongTensor(np.array(self.actions)),
            torch.FloatTensor(np.array(self.log_probs)),
            torch.FloatTensor(np.array(self.values)),  
            torch.FloatTensor(np.array(self.rewards)),
            torch.FloatTensor(np.array(self.dones)),
        )
    
    def __len__(self):
        return len(self.states)

In [13]:
class PPOAgent:
    def __init__(
        self,
        obs_dim:       int,
        n_actions:     int,
        lr:            float = 3e-4,
        gamma:         float = 0.99,    # discount factor
        lam:           float = 0.95,    # GAE lambda (bias-variance tradeoff)
        clip_eps:      float = 0.2,     # PPO clip range ε
        epochs:        int   = 10,      # gradient passes over each rollout
        batch_size:    int   = 64,      # mini-batch size within each epoch
        entropy_coef:  float = 0.01,    # c₂ — exploration encouragement
        value_coef:    float = 0.5,     # c₁ — critic loss weight
        rollout_steps: int   = 2048,    # collect this many steps before updating
    ):
        self.gamma         = gamma
        self.lam           = lam
        self.clip_eps      = clip_eps
        self.epochs        = epochs
        self.batch_size    = batch_size
        self.entropy_coef  = entropy_coef
        self.value_coef    = value_coef
        self.rollout_steps = rollout_steps
 
        self.ac        = ActorCritic(obs_dim, n_actions)
        self.optimizer = optim.Adam(self.ac.parameters(), lr=lr)
        self.buffer    = RolloutBuffer()
    
    # gae - generalized advantage estimation
    def compute_gae(self, rewards, values, dones, next_value: float):
        adavantages = []
        gae = 0.0
        values_ext = values + [next_value]

        for t in reversed(range(len(rewards))):
            delta = (rewards[t] + self.gamma*values_ext[t+1]*(1-dones[t]) - values_ext[t])
            gae = delta + self.gamma*self.lam*gae*(1-dones[t])
            adavantages.insert(0, gae)
        
        returns = [a+v for a,v in zip(adavantages, values)]
        return adavantages, returns

    def train(self):
        # ✅ FIX 1: guard against buffers too small to train on
        if len(self.buffer) < 2:
            self.buffer.clear()
            return

        states, actions, rewards, old_log_probs, values, dones = self.buffer.get()

        with torch.no_grad():
            _, next_val = self.ac(states[-1].unsqueeze(0))
            next_value  = float(next_val.item())

        advantages, returns = self.compute_gae(
            rewards.tolist(), values.tolist(), dones.tolist(), next_value
        )
        advantages = torch.FloatTensor(advantages)
        returns    = torch.FloatTensor(returns).detach()   # ✅ FIX 2: explicit detach — targets never need grad

        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        n = len(states)
        
        self.ac.train()   # ✅ FIX 3: ensure network is in training mode before backward()
        
        for _ in range(self.epochs):
            indices = torch.randperm(n)
            for start in range(0, n, self.batch_size):
                idx = indices[start : start + self.batch_size]

                log_probs, vals, entropy = self.ac.evaluate(states[idx], actions[idx])

                ratio = torch.exp(log_probs - old_log_probs[idx].detach())  # ✅ old_log_probs are targets too

                adv   = advantages[idx]
                surr1 = ratio * adv
                surr2 = torch.clamp(ratio, 1.0 - self.clip_eps, 1.0 + self.clip_eps) * adv
                actor_loss  = -torch.min(surr1, surr2).mean()

                critic_loss  = F.mse_loss(vals, returns[idx])
                entropy_loss = -entropy.mean()

                loss = actor_loss + self.value_coef * critic_loss + self.entropy_coef * entropy_loss

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.ac.parameters(), 0.5)
                self.optimizer.step()

        self.buffer.clear()

In [17]:
def train_ppo(env, agent: PPOAgent, total_timesteps: int, seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
 
    obs, _ = env.reset(seed=seed)
    state  = obs.flatten()
 
    pbar = tqdm(total=total_timesteps, desc='PPO  training')
    for t in range(total_timesteps):
        action, log_prob, value          = agent.ac.get_action(state)
        try:
            next_obs, reward, term, trunc, _ = env.step(action)
            done                             = term or trunc
        except IndexError:
            next_obs, _ = env.reset()
            reward, done = 0.0, True
            
        next_state                       = next_obs.flatten()
 
        agent.buffer.push(state, action, reward, log_prob, value, done)
        state = next_state if not done else env.reset()[0].flatten()
        pbar.update(1)
 
        # Trigger update when rollout buffer is full OR episode ends
        if len(agent.buffer) >= agent.rollout_steps or done:
            agent.train()
    pbar.close()

In [20]:
def test_agent(env, agent, agent_type: str, n_episodes: int, seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    rewards = []
 
    for _ in tqdm(range(n_episodes), desc=f'Testing {agent_type:4s}'):
        obs, _       = env.reset()
        state        = obs.flatten()
        total_reward = 0.0
        done         = False
 
        while not done:
            if agent_type == 'DQN':
                with torch.no_grad():
                    action = agent.q_net(
                        torch.FloatTensor(state).unsqueeze(0)
                    ).argmax().item()
            else:  
                action, _, _ = agent.ac.get_action(state)
 
            try:
                obs, reward, term, trunc, _ = env.step(action)
                done = term or trunc
            except IndexError:
                reward, done = 0.0, True
                obs, _ = env.reset()
                
            state         = obs.flatten()
            total_reward += float(reward)
 
        rewards.append(total_reward)
    return rewards

In [22]:
def print_stats(reward_over_episodes):
    """  Print Reward  """

    avg = np.mean(reward_over_episodes)
    min = np.min(reward_over_episodes)
    max = np.max(reward_over_episodes)

    print (f'Min. Reward          : {min:>10.3f}')
    print (f'Avg. Reward          : {avg:>10.3f}')
    print (f'Max. Reward          : {max:>10.3f}')

    return min, avg, max

In [23]:
SEED              = 42
TOTAL_EPISODES    = 50
TOTAL_TIMESTEPS   = 25_000
 
env = gym.make('stocks-v0')
obs, _ = env.reset()
obs_dim   = int(np.prod(obs.shape))
n_actions = env.action_space.n
plot_data     = {'x': list(range(1, TOTAL_EPISODES + 1))}
plot_settings = {}

ppo_agent = PPOAgent(obs_dim, n_actions)
train_ppo(env, ppo_agent, TOTAL_TIMESTEPS, SEED)
ppo_rewards               = test_agent(env, ppo_agent, 'PPO', TOTAL_EPISODES, SEED)
plot_data['PPO']          = ppo_rewards
plot_settings['PPO']      = f'Avg {np.mean(ppo_rewards):>7.2f} : PPO  25K'






























































































































































































































PPO  training: 100%|██████████| 25000/25000 [00:19<00:00, 1262.31it/s]






































































































Testing PPO : 100%|██████████| 50/50 [00:47<00:00,  1.04it/s]


In [25]:
print_stats(ppo_rewards)

Min. Reward          :     -5.120
Avg. Reward          :     -0.026
Max. Reward          :      3.629


(np.float64(-5.1199951171875),
 np.float64(-0.0261236572265625),
 np.float64(3.628631591796875))

In [28]:
print(len(ppo_rewards))

50
